# Chapter 9: Retrieval — Putting It to Work
## Topic 4: Should Retrieval Touch Customer Records Directly?

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every retrieval example built across Chapters 7-9 so far has searched over `fd_master_database.csv`-style *policy* content — FAQ answers, SOPs, product terms — content that is the same for every customer. `fd_master_database.csv` itself is a fundamentally different kind of data: individual customer records containing names, phone numbers, account numbers, and FD balances. This topic exists to ask directly: should the same semantic/hybrid retrieval machinery built for policy documents ever be pointed at this table?
- The short answer the theory arrives at is no, and this topic exists specifically to explain *why*, precisely, rather than leaving it as an unstated assumption. The reasoning connects directly to two things already established in this notebook series: Chapter 7 Topic 2 demonstrated with real numbers that semantic search can confuse two *different customers'* FD reference numbers (`BJ2019FD7717` vs `XY2019FD7717`) purely because they share character fragments, and Chapter 7 Topic 8 established that metadata filtering used for access control must always be pre-filtering, never post-filtering, precisely because post-filtering means unauthorized records were already fetched before being discarded.
- The core distinction this topic formalizes: **policy retrieval** answers "what does the rule say" and is the same for every customer — genuinely well-suited to semantic search, since the goal is finding the most *relevant* explanation. **Record lookup** answers "what is true about this specific, identified customer" and has exactly one correct answer per query, identified by an exact key (an FD reference number, an account number) — a lookup problem, not a relevance-ranking problem, and semantic search's entire value proposition (finding approximately-relevant content when there's no exact match) is actively the wrong tool here.


### 2. Internal Working — Step by Step

**Why record lookup is architecturally different from policy retrieval, step by step:**

1. **The retrieval question is different in kind, not just degree.** Policy retrieval asks "which of these documents is most relevant to this query" — a genuinely fuzzy question with a best-effort answer. Record lookup asks "give me the row where `FD_No = 'BJ2019FD7717'`" — a question with exactly one correct answer, or none at all, never a "close enough" answer.
2. **Semantic similarity between two different customers' records is a measured, demonstrated risk, not a hypothetical one.** Chapter 7 Topic 2's own executed code showed `BJ2019FD7717` and `XY2019FD7717` scoring 0.98 cosine similarity via character-fragment overlap — despite referring to two entirely different accounts. If record lookup used the same embedding-based approach as policy retrieval, this is exactly the mechanism by which one customer's balance, phone number, or maturity date could surface in response to a query about a *different* customer.
3. **The correct mechanism is exact-match database lookup, not similarity search.** Once a customer identifier (an FD reference number, validated by the project's existing `validate_fd_reference` tool from Chapter 3) is confirmed, retrieving that customer's record is a direct, deterministic key-based lookup — `WHERE FD_No = ?` — with no ranking, no top-k, no similarity threshold involved anywhere in the process.
4. **Where the two mechanisms can legitimately meet:** a single customer-facing answer might need *both* — a specific fact about this customer's own record (exact lookup) *and* a general policy explanation of what that fact means (semantic retrieval over policy documents). These are two separate tool calls with two separate mechanisms, composed together in the final answer — never a single retrieval call blending both kinds of data into one index.


### 3. How This Is Implemented in This Project

- `validate_fd_reference` (Chapter 3's existing tool) already implements the correct pattern for this exact problem: given a reference number extracted from the email, it performs a deterministic, exact lookup against `fd_master_database.csv` — not a similarity search, not a ranked retrieval call. This tool already embodies this topic's central architectural conclusion; this topic exists to make that design choice explicit and generalizable, rather than leaving it as an implicit detail of one specific tool.
- The retrieval pipeline built across Chapter 7 (BM25, dense retrieval, hybrid RRF, reranking) is correctly scoped to the *policy* knowledge base only (the FAQ/SOP/product-guide chunks) — `fd_master_database.csv` should never be embedded into the same vector index, or searched via the same `search_knowledge_base` tool wired in Topic 3.
- When an answer genuinely needs both kinds of information — for example, "what is my exact penalty amount if I withdraw today" needs this specific customer's FD amount and booking date (exact lookup via `validate_fd_reference`) *combined with* the general penalty policy rate (semantic retrieval via `search_knowledge_base`) — the agent calls both tools separately in the same turn, and the final answer synthesizes both results. Chapter 8's citation mechanism should distinguish these two source types explicitly (a database record vs a policy document) rather than treating them as interchangeable "sources."


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **The cross-customer leak risk is the central security concern, and it's already measured, not speculative.** Chapter 7 Topic 2's executed code demonstrated a 0.98 similarity score between two entirely different customers' reference numbers — if this mechanism were ever used for actual record retrieval instead of exact lookup, this is a directly demonstrated path to one customer's account details leaking into another customer's conversation.
- **PII exposure surface:** every field in `fd_master_database.csv` — customer name, mobile number, account number, FD amount — is PII. Any mechanism that embeds this data into a vector index (even for a legitimate-seeming reason, like "let the model semantically search customer notes") multiplies the PII exposure surface, since the embeddings themselves can potentially leak information about the underlying text (a concern already raised for policy content in Chapter 7 Topic 2, and far more consequential here given the sensitivity of the underlying data).
- **Debugging a "wrong customer's data displayed" incident:** if this ever happens, the root-cause investigation should immediately check whether record retrieval used exact-match lookup or any form of similarity/ranking-based retrieval — given the measured cross-customer similarity risk, any non-exact-match mechanism touching customer records is the primary suspect.
- **Latency and cost:** exact-match database lookup (`WHERE FD_No = ?`) is dramatically faster and cheaper than embedding a query and running similarity search — another practical, non-security reason record lookup should never be implemented via the semantic retrieval pipeline even before considering the safety argument.
- **Monitoring:** any tool or code path that touches `fd_master_database.csv` should be logged with the exact identifier used for the lookup and the verification step that confirmed it (matching the existing `validate_fd_reference` audit pattern) — this creates an auditable trail distinct from policy-retrieval logging, appropriate for the more sensitive nature of the data involved.
- **Access scoping beyond exact-match itself:** even with correct exact-match lookup, the calling context matters — a lookup should only ever be performed using an identifier that was actually verified as belonging to the customer on the current conversation (via the existing validation tool), never an identifier the model might construct or guess from partial information in the email.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Unifying policy and customer-record retrieval into one system vs keeping them structurally separate:** a unified system (one vector index, one search tool covering both) might seem simpler to build and reason about, but this topic's entire argument is that the two have fundamentally different correctness requirements — relevance-ranking is the wrong model for a problem that has exactly one correct answer per identified customer. Keeping them as two structurally distinct mechanisms (semantic retrieval for policy, exact lookup for records) is the safer and more architecturally honest choice, even though it means maintaining two separate code paths.
- **Should customer-specific free-text notes (if they existed) ever be semantically searchable?** if a support system stored free-text case notes per customer, there could be a legitimate case for semantic search *within* that specific customer's own notes, scoped by an exact customer-identifier pre-filter (Chapter 7 Topic 8's pre-filtering pattern) — this is different from semantically searching *across* all customers' structured records, which is the specific pattern this topic argues against.
- **The trade-off between engineering convenience and demonstrated security risk:** it might be tempting to reuse the same well-tested `search_knowledge_base` infrastructure for "consistency," but Chapter 7 Topic 2's own measured 0.98 cross-customer similarity finding is direct, already-available evidence against this convenience — this is a case where a real measurement should override an appealing simplification.


### 6. Alternatives and When to Use Each

- **Exact-match database lookup (the correct approach for customer records):** the right mechanism whenever a query is about a specific, already-identified customer's own data — this is what `validate_fd_reference` already correctly implements.
- **Semantic/hybrid retrieval (the correct approach for policy content):** the right mechanism whenever a query is about general rules, procedures, or product information that's the same regardless of which customer is asking.
- **Semantic search scoped to a single customer's own free-text content, pre-filtered by verified identity:** a legitimate middle case, if such content exists in a system's design — the pre-filter must be based on a verified identifier, never inferred or guessed, and must never search across other customers' content.
- **Never: semantically searching across all customers' structured records:** this is the specific pattern this topic argues against, given the measured cross-customer similarity risk and the fact that this kind of query always has an exact, identifiable correct answer rather than a ranked "most relevant" one.


### 7. Common Mistakes and Production Failures

- Embedding customer records into the same vector index used for policy documents, "for consistency" or engineering convenience, without accounting for the demonstrated cross-customer similarity risk this creates.
- Treating "which record is most similar to this query" as an acceptable substitute for "which record exactly matches this verified identifier" — these are different problems with different correctness guarantees, and conflating them is precisely the mistake this topic exists to prevent.
- Allowing a model to construct or guess a customer identifier from partial information in an email, rather than requiring the identifier to be explicitly validated (via `validate_fd_reference`) before any record lookup occurs.
- Logging policy-retrieval and customer-record-lookup calls identically, losing the more rigorous audit trail that the sensitivity of customer PII warrants.
- Not recognizing that a single customer-facing answer needing both record-specific and policy-general information should combine two separate tool calls with two separate mechanisms, rather than trying to force both into one retrieval call.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why shouldn't customer records be retrieved using the same semantic search mechanism used for policy documents?
  A: Policy retrieval is a relevance-ranking problem — the same rules apply to every customer, and finding the "most relevant" explanation is a reasonable goal. Customer record lookup is a different kind of problem — it has exactly one correct answer per verified identifier, and semantic similarity between two different customers' records (measurably demonstrated in Chapter 7 Topic 2, where two different reference numbers scored 0.98 similarity due to shared character fragments) creates a real risk of one customer's data surfacing for a different customer's query.

- Q: What mechanism should be used instead for customer record retrieval?
  A: Exact-match database lookup using a verified identifier — for example, `WHERE FD_No = ?` — with no ranking, no top-k, and no similarity threshold involved. This is exactly what the project's existing `validate_fd_reference` tool already implements.

**Intermediate**

- Q: A teammate proposes embedding `fd_master_database.csv` into the same vector index as the policy knowledge base, arguing it would let customers ask natural-language questions about their own records. How do you respond?
  A: The stated goal (natural-language questions) doesn't require semantic search across *all* customer records — it requires exact lookup of *this specific, already-identified* customer's record, followed by natural-language generation over that looked-up data. Embedding all customer records into one shared vector index reintroduces the measured cross-customer similarity risk from Chapter 7 Topic 2, multiplies the PII exposure surface, and is also simply slower and more expensive than a direct database lookup for a problem that has an exact answer.

- Q: How should an answer that needs both a specific customer's data and general policy information be constructed?
  A: As two separate tool calls using two separate mechanisms — an exact-match lookup (via `validate_fd_reference` or equivalent) for the customer-specific fact, and a semantic/hybrid retrieval call (via `search_knowledge_base`) for the general policy explanation — composed together in the final generated answer, with citation distinguishing which source each part of the answer relies on.

**Advanced**

- Q: Design the safeguards needed to ensure a customer-record lookup mechanism never surfaces the wrong customer's data.
  A: First, the lookup must use exact-match database query logic, never any form of similarity or ranking-based retrieval, eliminating the fuzzy-matching risk category entirely. Second, the identifier used for the lookup must come from a verified source — validated against the customer's own submitted information via the existing validation tool, never guessed or inferred by the model from partial or ambiguous information in the email. Third, every such lookup should be logged with the identifier used and the verification step that confirmed it, creating an auditable trail specifically for this more sensitive data category, distinct from and more rigorous than general policy-retrieval logging.

- Q: How does this topic's conclusion connect to Chapter 7 Topic 8's discussion of metadata filtering for access control?
  A: Topic 8 established that access-control filtering must always be pre-filtering (restricting the search space before searching) rather than post-filtering (searching everything, then discarding unauthorized results afterward), because post-filtering means unauthorized records were already fetched and processed before being discarded — a real security risk if any bug exists in the discard logic. This topic's conclusion is a more extreme version of the same principle: for customer records specifically, the safest design isn't just pre-filtering within a shared search mechanism, it's using an entirely different mechanism (exact lookup) that structurally cannot return any record other than the one explicitly requested, removing the entire class of risk rather than mitigating it with a filter.

**Scenario-based**

- Q: A production incident report shows a customer received a response mentioning another customer's FD amount. Walk through your investigation.
  A: Immediately check whether the code path that produced this response used exact-match lookup or any similarity/ranking-based retrieval to access customer records — given the demonstrated cross-customer similarity risk, this is the primary suspect. Check the specific identifier used for the lookup and whether it was properly verified before the lookup occurred, or whether it was inferred or guessed from ambiguous information in the email. Audit whether `fd_master_database.csv` or any customer-record data was ever embedded into a shared vector index alongside policy content, which would be the clearest structural cause of this kind of leak.

**Follow-up questions to expect:**

- "What would you do differently if the customer had genuinely lost or forgotten their FD reference number?"
- "How would you extend this reasoning to a system that also stores free-text customer support notes?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic is a direct, concrete application of a general access-control principle already established in Chapter 7 Topic 8, taken to its logical conclusion:** pre-filtering is safer than post-filtering for access control; using an entirely different, structurally incapable-of-leaking mechanism (exact lookup) is safer still, when the data sensitivity and the nature of the query (exactly-one-correct-answer) both support it.
- **The "relevance ranking vs exact lookup" distinction generalizes well beyond this specific project:** any system mixing "find the most relevant thing" content with "find the exact, uniquely-identified thing" content faces this same architectural question — recognizing which category a given piece of data falls into is a broadly reusable design skill.
- **This topic connects retrieval architecture directly to data governance and compliance concerns:** in a regulated financial domain, decisions like this one aren't purely technical — they have direct implications for data protection compliance, and being able to articulate the technical reasoning in terms a compliance stakeholder would recognize (exact lookup vs fuzzy matching, audit trails, PII exposure surface) is a genuinely valuable, senior-level communication skill.

### 10. Quick Revision Summary (for last-minute interview prep)

> Policy retrieval and customer-record retrieval are fundamentally different problems and must use fundamentally different mechanisms. Policy retrieval is a relevance-ranking problem — the same content applies to every customer, and semantic/hybrid retrieval (Chapter 7's full pipeline) is well-suited to it. Customer-record retrieval is an exact-lookup problem — every query has exactly one correct answer per verified identifier, and Chapter 7 Topic 2's own measured finding (two different customers' FD reference numbers scoring 0.98 similarity due to shared character fragments) demonstrates a real, not hypothetical, risk of cross-customer data leakage if similarity-based retrieval were ever used for this purpose instead of exact-match database lookup. The project's existing `validate_fd_reference` tool already correctly implements this principle. When an answer needs both a specific customer's data and general policy context, the correct architecture is two separate tool calls using two separate mechanisms, composed together in the final answer — never one shared retrieval system blending both kinds of data into a single index.


### Module 1: The Correct Mechanism — Exact-Match Lookup on Real Customer Records

Using real rows shaped exactly like the project's actual `fd_master_database.csv`, implement the correct exact-lookup mechanism, mirroring `validate_fd_reference`.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Exact-match lookup on real-shaped customer records
#
# These rows use the SAME schema and SAME style of realistic values
# as this project's actual fd_master_database.csv (FD_No, Customer_Name,
# Mobile_Number, Account_Number, FD_Amount_INR, etc.)
# ------------------------------------------------------------------

CUSTOMER_RECORDS = [
    {"FD_No": "BJ2019FD7717", "Customer_Name": "Shobha Chopra", "Mobile_Number": "9686667744",
     "FD_Amount_INR": 391000, "Interest_Rate_Percent": 6.86, "Status": "Closed (Premature)"},
    {"FD_No": "BJ2022FD6918", "Customer_Name": "Anita Mishra", "Mobile_Number": "9449835534",
     "FD_Amount_INR": 160000, "Interest_Rate_Percent": 7.39, "Status": "Active"},
    {"FD_No": "XY2019FD7717", "Customer_Name": "Rajesh Kumar", "Mobile_Number": "9876543210",
     "FD_Amount_INR": 275000, "Interest_Rate_Percent": 7.10, "Status": "Active"},
    {"FD_No": "BJ2023FD4792", "Customer_Name": "Sunita Jain", "Mobile_Number": "8316079859",
     "FD_Amount_INR": 456000, "Interest_Rate_Percent": 7.08, "Status": "Closed (Premature)"},
]


def validate_fd_reference(reference_number: str) -> dict:
    """The CORRECT mechanism -- exact-match lookup, mirroring this
    project's real existing validate_fd_reference tool. No ranking,
    no similarity threshold, no 'top-k' -- either the exact key
    matches or it doesn't."""
    for record in CUSTOMER_RECORDS:
        if record["FD_No"] == reference_number:
            return {"found": True, "record": record}
    return {"found": False, "record": None}


print("=" * 70)
print("MODULE 1: EXACT-MATCH LOOKUP (the correct mechanism)")
print("=" * 70)

test_lookups = ["BJ2019FD7717", "XY2019FD7717", "BJ9999FD0000"]
for ref in test_lookups:
    result = validate_fd_reference(ref)
    print(f"\nLookup: '{ref}'")
    if result["found"]:
        r = result["record"]
        name = r["Customer_Name"]
        amount = r["FD_Amount_INR"]
        status = r["Status"]
        print(f"  FOUND -- Customer: {name}, Amount: Rs {amount:,}, Status: {status}")
    else:
        print("  NOT FOUND -- no such reference number exists.")

print("\nNotice BJ2019FD7717 and XY2019FD7717 return COMPLETELY DIFFERENT")
print("customers, correctly, despite looking superficially similar as")
print("strings -- exact-match lookup has NO mechanism by which these two")
print("could ever be confused with each other.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: EXACT-MATCH LOOKUP (the correct mechanism)

Lookup: 'BJ2019FD7717'
  FOUND -- Customer: Shobha Chopra, Amount: Rs 391,000, Status: Closed (Premature)

Lookup: 'XY2019FD7717'
  FOUND -- Customer: Rajesh Kumar, Amount: Rs 275,000, Status: Active

Lookup: 'BJ9999FD0000'
  NOT FOUND -- no such reference number exists.

Notice BJ2019FD7717 and XY2019FD7717 return COMPLETELY DIFFERENT
customers, correctly, despite looking superficially similar as
strings -- exact-match lookup has NO mechanism by which these two
could ever be confused with each other.

Module 1 complete. Run Module 2 next.


### Module 2: The Risky Alternative — What Semantic Search Would Do Instead

Rebuild Chapter 7 Topic 2's exact cross-customer similarity demonstration, now applied directly to these real-shaped customer records, to make the risk concrete rather than abstract.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: What semantic search would do if used for record lookup
#
# This reuses the EXACT mechanism from Chapter 7 Topic 2 (char n-gram
# TF-IDF + SVD) that measurably confused BJ2019FD7717 and XY2019FD7717
# at 0.98 similarity -- now shown directly against REAL customer
# records to make the risk concrete, not just abstract.
# ------------------------------------------------------------------

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def normalize_vector(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def cosine_similarity(a, b):
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / d) if d > 0 else 0.0


reference_numbers = [r["FD_No"] for r in CUSTOMER_RECORDS]

# char n-grams, exactly as in Chapter 7 Topic 2 -- this is what makes
# identifiers look deceptively similar based on shared fragments
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
sparse = vectorizer.fit_transform(reference_numbers)
svd = TruncatedSVD(n_components=3, random_state=42)
dense = svd.fit_transform(sparse)
dense_norm = np.array([normalize_vector(v) for v in dense])

query_ref = "BJ2019FD7717"
query_idx = reference_numbers.index(query_ref)
query_vec = dense_norm[query_idx]

print("=" * 70)
print("MODULE 2: WHAT SEMANTIC SEARCH WOULD RETURN (the risky alternative)")
print("=" * 70)
print(f"Simulated semantic lookup for: '{query_ref}'\n")

similarities = [(ref, cosine_similarity(query_vec, dense_norm[i]))
                for i, ref in enumerate(reference_numbers)]
similarities.sort(key=lambda x: x[1], reverse=True)

for ref, sim in similarities:
    record = next(r for r in CUSTOMER_RECORDS if r["FD_No"] == ref)
    match_flag = "EXACT MATCH" if ref == query_ref else "DIFFERENT CUSTOMER"
    customer_name = record["Customer_Name"]
    print(f"  {ref:<16} | similarity={sim:.4f} | {match_flag:<20} | {customer_name}")

top_result = similarities[0][0]
second_result = similarities[1] if len(similarities) > 1 else None

print(f"\nIf record retrieval used 'return the top-1 most similar record'")
print(f"logic instead of exact-match, and a bug or threshold issue caused")
print(f"the SECOND-ranked result to be returned instead of exact match")
print(f"(e.g. from a mis-set top-k, or a fuzzy-match fallback):")
if second_result:
    wrong_ref, wrong_sim = second_result
    wrong_record = next(r for r in CUSTOMER_RECORDS if r["FD_No"] == wrong_ref)
    wrong_name = wrong_record["Customer_Name"]
    query_customer_name = CUSTOMER_RECORDS[query_idx]["Customer_Name"]
    print(f"  -> Could surface '{wrong_ref}' (similarity={wrong_sim:.4f}),")
    print(f"     belonging to {wrong_name} -- a COMPLETELY")
    print(f"     DIFFERENT customer's amount, phone number, and status,")
    print(f"     returned for a query about {query_customer_name}'s account.")

print("\nThis is not a hypothetical -- this is the SAME mechanism and the")
print("SAME magnitude of risk measured with real numbers in Chapter 7")
print("Topic 2, now applied directly to realistic customer record data.")
print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: WHAT SEMANTIC SEARCH WOULD RETURN (the risky alternative)
Simulated semantic lookup for: 'BJ2019FD7717'

  BJ2019FD7717     | similarity=1.0000 | EXACT MATCH          | Shobha Chopra
  XY2019FD7717     | similarity=0.9847 | DIFFERENT CUSTOMER   | Rajesh Kumar
  BJ2023FD4792     | similarity=0.1467 | DIFFERENT CUSTOMER   | Sunita Jain
  BJ2022FD6918     | similarity=0.1467 | DIFFERENT CUSTOMER   | Anita Mishra

If record retrieval used 'return the top-1 most similar record'
logic instead of exact-match, and a bug or threshold issue caused
the SECOND-ranked result to be returned instead of exact match
(e.g. from a mis-set top-k, or a fuzzy-match fallback):
  -> Could surface 'XY2019FD7717' (similarity=0.9847),
     belonging to Rajesh Kumar -- a COMPLETELY
     DIFFERENT customer's amount, phone number, and status,
     returned for a query about Shobha Chopra's account.

This is not a hypothetical -- this is the SAME mechanism and the
SAME magnitude of risk measured with real 

### Module 3: The Correct Composed Pattern — Two Mechanisms, One Answer

Demonstrates the theory's recommended architecture: a customer-specific fact via exact lookup, combined with general policy context via semantic retrieval, as two separate tool calls.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Composing exact lookup + semantic retrieval correctly
# ------------------------------------------------------------------

from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    return text.lower().split()

# the POLICY knowledge base -- structurally SEPARATE from customer
# records, never mixed into the same index
POLICY_KNOWLEDGE_BASE = [
    {"text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.", "source": "01_FD_FAQ.pdf"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.", "source": "02_FD_Product_Guide.pdf"},
]
policy_bm25 = BM25Okapi([tokenize(doc["text"]) for doc in POLICY_KNOWLEDGE_BASE])


def search_knowledge_base(query: str, top_k: int = 1) -> list:
    """Policy retrieval -- semantic/hybrid, appropriate for THIS kind
    of content (general rules, same for every customer)."""
    scores = policy_bm25.get_scores(tokenize(query))
    ranked = sorted(range(len(POLICY_KNOWLEDGE_BASE)), key=lambda i: scores[i], reverse=True)
    return [{"text": POLICY_KNOWLEDGE_BASE[i]["text"], "source": POLICY_KNOWLEDGE_BASE[i]["source"]}
            for i in ranked[:top_k]]


def answer_customer_query(customer_fd_reference: str, question: str) -> dict:
    """The CORRECT composed pattern: exact lookup for the customer's
    own data, semantic retrieval for general policy context -- two
    SEPARATE mechanisms, two separate tool calls, combined in the
    final answer with distinct source attribution."""
    lookup_result = validate_fd_reference(customer_fd_reference)   # EXACT lookup
    policy_result = search_knowledge_base(question)                 # SEMANTIC retrieval

    if not lookup_result["found"]:
        return {"error": "Could not verify customer FD reference -- cannot proceed."}

    record = lookup_result["record"]
    policy_chunk = policy_result[0] if policy_result else None

    return {
        "customer_specific_fact": {
            "source_type": "EXACT_DATABASE_LOOKUP",
            "fd_amount": record["FD_Amount_INR"],
            "fd_status": record["Status"],
        },
        "general_policy_context": {
            "source_type": "SEMANTIC_RETRIEVAL",
            "source": policy_chunk["source"] if policy_chunk else None,
            "text": policy_chunk["text"] if policy_chunk else None,
        },
    }


print("=" * 70)
print("MODULE 3: CORRECT COMPOSED PATTERN -- exact lookup + semantic retrieval")
print("=" * 70)

result = answer_customer_query("BJ2019FD7717", "what is the penalty for premature withdrawal")

customer_fact = result["customer_specific_fact"]
policy_context = result["general_policy_context"]

print(f"\nCustomer-specific fact (via {customer_fact['source_type']}):")
print(f"  FD Amount: Rs {customer_fact['fd_amount']:,}")
print(f"  Status: {customer_fact['fd_status']}")

print(f"\nGeneral policy context (via {policy_context['source_type']}):")
print(f"  Source: {policy_context['source']}")
print(f"  Text: {policy_context['text'][:70]}...")

print("\nTwo DIFFERENT mechanisms, two DIFFERENT source-type tags, composed")
print("into one answer -- never one shared retrieval call blending both")
print("kinds of data into a single index.")

print("\nModule 3 complete. All Chapter 9 Topic 4 modules done.")
print()
print("=" * 70)
print("CHAPTER 9 TOPIC 4 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Customer records = EXACT-MATCH lookup only. Never similarity search,
  never ranking, never a "top-k" over customer records.

  Policy content = semantic/hybrid retrieval (Chapter 7's full
  pipeline) -- the same content applies to every customer, and finding
  the "most relevant" explanation is a reasonable goal.

  The cross-customer similarity risk is MEASURED, not hypothetical --
  Chapter 7 Topic 2 showed 0.98 similarity between two different
  customers' reference numbers from shared character fragments alone.

  When an answer needs both: TWO separate tool calls, TWO separate
  mechanisms, combined in the final answer with distinct source
  attribution -- never one shared index mixing both data types.
""")


MODULE 3: CORRECT COMPOSED PATTERN -- exact lookup + semantic retrieval

Customer-specific fact (via EXACT_DATABASE_LOOKUP):
  FD Amount: Rs 391,000
  Status: Closed (Premature)

General policy context (via SEMANTIC_RETRIEVAL):
  Source: 01_FD_FAQ.pdf
  Text: Premature withdrawal of FD incurs a 1 percent penalty on the applicabl...

Two DIFFERENT mechanisms, two DIFFERENT source-type tags, composed
into one answer -- never one shared retrieval call blending both
kinds of data into a single index.

Module 3 complete. All Chapter 9 Topic 4 modules done.

CHAPTER 9 TOPIC 4 -- KEY POINTS TO REMEMBER

  Customer records = EXACT-MATCH lookup only. Never similarity search,
  never ranking, never a "top-k" over customer records.

  Policy content = semantic/hybrid retrieval (Chapter 7's full
  pipeline) -- the same content applies to every customer, and finding
  the "most relevant" explanation is a reasonable goal.

  The cross-customer similarity risk is MEASURED, not hypothetical --
  Chapt